In [1]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.nn.modules.conv import Conv, autopad
import ultralytics.nn.tasks as yolo_tasks
from ultralytics.nn.modules import Conv



In [2]:
class GatedFusion(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c * 2, c, 1),
            nn.Sigmoid()
        )

    def forward(self, x1, x2):
        g = self.gate(torch.cat([x1, x2], dim=1))
        self.last_gate = g.detach()   # <-- critical
        return g * x1 + (1 - g) * x2


In [3]:
class FirstConvGatedConv(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, g=1, act=True):
        super().__init__()

        # DO NOT pass p=None
        self.rgb_conv = Conv(3, c2, k=3, s=2, p=1, g=g, act=act)
        self.range_conv = Conv(1, c2, k=3, s=2, p=1, g=g, act=act)


        self.fusion = GatedFusion(c2)

    def forward(self, x):
        # Stride inference path (Ultralytics uses 3ch dummy tensor)
        if x.shape[1] == 3:
            return self.rgb_conv(x)

        rgb = x[:, :3]
        rng = x[:, 3:4]

        frgb = self.rgb_conv(rgb)
        frng = self.range_conv(rng)

        return self.fusion(frgb, frng)

In [4]:
yolo_tasks.FirstConvGatedConv = FirstConvGatedConv

globals()["FirstConvGatedConv"] = FirstConvGatedConv
yolo = YOLO("yolo11n_gated.yaml")
print("YOLO11n with gated early fusion loaded successfully")


YOLO11n with gated early fusion loaded successfully


In [5]:
yolo = YOLO("yolo11n_gated.yaml")

dummy = torch.randn(1, 4, 1024, 1024)
with torch.no_grad():
    _ = yolo.model(dummy)

print("✅ YOLO11 gated fusion model is valid")


✅ YOLO11 gated fusion model is valid


In [6]:
# gate_buffer = []

# def gate_collect_hook(module, inp, out):
#     if hasattr(module, "last_gate"):
#         g = module.last_gate
#         gate_buffer.append((
#             g.mean().item(),
#             g.std().item()
#         ))

# for m in yolo.model.modules():
#     if isinstance(m, GatedFusion):
#         m.register_forward_hook(gate_collect_hook)

# def on_train_epoch_end(trainer):
#     if len(gate_buffer) == 0:
#         return

#     means = [m for m, _ in gate_buffer]
#     stds  = [s for _, s in gate_buffer]

#     print(
#         f"[Epoch {trainer.epoch}] "
#         f"GATE mean={sum(means)/len(means):.4f}, "
#         f"std={sum(stds)/len(stds):.4f}"
#     )

#     gate_buffer.clear()

# yolo.add_callback("on_train_epoch_end", on_train_epoch_end)


In [7]:
import torch

device = torch.device("cuda:0")

yolo.model = yolo.model.to(device)
yolo.model.eval()


DetectionModel(
  (model): Sequential(
    (0): FirstConvGatedConv(
      (rgb_conv): Conv(
        (conv): Conv2d(3, 3, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(3, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (range_conv): Conv(
        (conv): Conv2d(1, 3, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(3, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (fusion): GatedFusion(
        (gate): Sequential(
          (0): AdaptiveAvgPool2d(output_size=1)
          (1): Conv2d(6, 3, kernel_size=(1, 1), stride=(1, 1))
          (2): Sigmoid()
        )
      )
    )
    (1): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act

#VERIFICATION

In [8]:
def gate_probe(module, inp, out):
    frgb, frng = inp
    f = torch.cat([frgb, frng], dim=1)  # (B, 6, 1, 1) after pooling
    with torch.no_grad():
        g = module.gate(f)
        print(
            "Gate mean:", g.mean().item(),
            "std:", g.std().item()
        )


for m in yolo.model.modules():
    if isinstance(m, GatedFusion):
        m.register_forward_hook(gate_probe)


In [9]:
from ultralytics.utils import LOGGER
gate_buffer = []
gate_stats = []

def gate_hook(module, inp, out):
    if hasattr(module, "last_gate"):
        g = module.last_gate
        gate_stats.append((
            g.mean().item(),
            g.std().item()
        ))

for m in yolo.model.modules():
    if m.__class__.__name__ == "GatedFusion":
        m.register_forward_hook(gate_hook)


def on_train_epoch_end(trainer):
    if not gate_stats:
        return

    means = [m for m, _ in gate_stats]
    stds  = [s for _, s in gate_stats]

    mean_gate = sum(means) / len(means)
    std_gate  = sum(stds)  / len(stds)

    LOGGER.info(
        f"[GATED FUSION] Epoch {trainer.epoch}: "
        f"mean={mean_gate:.4f}, std={std_gate:.4f}"
    )

    gate_stats.clear()
yolo.add_callback("on_train_epoch_end", on_train_epoch_end)


In [10]:
device = next(yolo.model.parameters()).device
print(device)
x = torch.randn(1, 4, 1024, 1024, device=device)
with torch.no_grad():
    _ = yolo.model(x)
print("Forward pass OK")


cuda:0
Gate mean: 0.48423776030540466 std: 0.0447489395737648
Forward pass OK


In [11]:
model = YOLO("yolo11n_gated.yaml")

model.train(
    data="3ch_replreflec.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,
    batch=8,
    device=0,
    project="4CH_GatedFusion",
    name="nearir_signal_reflec_range",
    amp=False,
    augment=False,
    workers=0,
)


New https://pypi.org/project/ultralytics/8.4.12 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=3ch_replreflec.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n_gated.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=nearir_sig

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001EC8F19AB10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480